# Exploratory data analysis in Python

For the first section of this practical we will use the [famous Iris dataset][iris_wiki].

<figure>
    <img src='https://miro.medium.com/max/2550/1*7bnLKsChXq94QjtAiRn40w.png' height=200 width=400/>
    <figcaption>
        Fig. 1: The iris dataset contains measurements of the length and width of sepals and petals for 150 flowers from 3 Iris species
    </figcaption>
</figure>

---


We will use several well known data science libraries: `pandas`, `numpy`, `scipy`, `scikitlearn`, and `seaborn`.

### Import python libraries

[iris_wiki]: https://en.wikipedia.org/wiki/Iris_flower_data_set

In [ ]:
# These are all libraries for the entire notebook
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn import datasets, cluster
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import pdist, squareform
from scipy.stats import pearsonr, spearmanr

RANDOM_STATE = 42

# 1. Correlation _(30 minutes)_

In this notebook, you will take a look at what correlation is, and how you can use it for exploratory data analysis.

#### Simulating data
A simple way to show correlation is to simulate some data for which we know how it is related. This data generating process goes as follows:

First select some uniformly distributed random points between 0 and 10

$$X1 \sim U(0, 10)$$

Then, add some uniformly distributed noise (between 0 and 1) to generate the second variable

$$X2 = X1 + U(0, 1)$$

In [ ]:
'''
numpy syntax for generating uniformly distibuted data is 
np.random.uniform(LOWER_BOUND, UPPER_BOUND, NUMBER_OF_SAMPLES)
'''
X1 = np.random.uniform(0, 10, 50) # 50 random points between 0 and 10
X2 = X1 + np.random.uniform(0, 1, 50) # 50 times random noise between 0 and 1
sns.scatterplot(x=X1, y=X2);

#### Pearson correlation
The above figure clearly looks correlated. By calculating the pearson correlation coefficient (often referred to as `pearson r`), we can get a number of _how_ correlated. The pearson correlation coefficient ranges from -1 to +1, where -1 indicates a strong negative correlation, and +1 indicates a strong positive correlation. In addition, it is possible to calculate a p-value for whether the correlation coefficient differs from zero.

Two important assumptions made by pearson correlation are that __both variables are normally distributed__, and that __the relationship between the variables is linear__. You will first look at some examples where these assumptions are met, and later explore some options for when these assumptions do not hold.

With the code below you can use `scipy` to calculate the pearson correlation coefficient (and p-value) for your simulated data.

In [ ]:
'''
scipy's pearsonr function returns two values: 
(correlation_coefficient, p_value)
'''
pearsonr(X1, X2)

> 1.1 What does pearson's r  and and the accompanying p-value tell you about the relationship between our simulated variables?

---
Answer:

---

Change the data generating process so that the noise level is increased.  
_hint: sample from $U(0,20)$ instead of $U(0,1)$_. 

> 1.2 What happens with pearson's r and the accompanying p-value?

---
Answer:

---

#### Calculating correlation on the iris dataset
You will now revisit the iris dataset to look at correlations between the different measurements. The first few steps should look familiar by now.

#### Load the iris dataset

In [ ]:
iris_df = pd.read_excel('../../datasets/iris_data.xlsx')
iris_df.head()

In [ ]:
iris_df.species = iris_df.species.astype('category')

#### Visually inspect pairwise scatterplots 

In [ ]:
sns.pairplot(iris_df, hue='species name')

Looking at the pair-plots, petal length and petal width seem correlated. Let's attach a value to that using pearson correlation.

In [ ]:
pearsonr(iris_df["petal length"], iris_df["petal width"])

> 1.3 What does the correlation coefficient and p-value tell you about the correlation between petal length and petal width?

---
Answer:

---

Now change the code above to calculate the pearson correlation between sepal length and sepal width?

> 1.4 Are sepal length and sepal width correlated?

---
Answer:

---

#### All-VS-all correlation
A quick way to summarize relationships between variables in a dataset is to calculate all possible pairwise correlations and visualize them in a heatmap.

In [ ]:
sns.heatmap(iris_df.corr(method='pearson'), vmin=-1, vmax=1, 
            center=0, annot=True, square=True)

> 1.5 What two variables are most strongly correlated?

---
Answer:

---

> 1.6 Why are there only ones (1) on the diagonal?

---
Answer:

---

> 1.7 What do you think of the color scale that was used for this figure?

---
Answer:

---

Add the following argument to the heatmap function call: `cmap="RdBu_r"` (make sure to separate arguments with a comma)

> 1.8 Is the new color scale an improvement over the previous one? Why (not)?

---
Answer:

---

### The perils of assuming linear relationships

As discussed previously, pearson correlation relies on the assumption that the relationship between two variables is linear. In the following section you will find out what happens when these assumptions are not met. To illustrate this, you will use a well know dataset called Anscombe's quartet.

#### Load dataset

In [ ]:
anscombe_df = pd.read_csv('../../datasets/anscombe_quartet.csv', header=[0,1])
anscombe_df.head()

#### Calculate some summary statistics

In [ ]:
anscombe_df.describe()

> 1.9 Can you distinguish the four datasets based on their summary statistics?

---
Answer:

---

Let's see how linearly correlated the points in each dataset are.

In [ ]:
for dataset in ['I','II','III','IV']:
    r, p_value = pearsonr(anscombe_df[dataset, 'x'], anscombe_df[dataset, 'y'])
    print(f'Dataset {dataset}: Pearson r = {round(r, 2)}, p value = {round(p_value, 2)}')

> 1.10 How correlated do you find these datasets?

---
Answer:

---

Based on the thing you looked at so far, you would probably conclude that these datasets are very similar, if not identical. You will now make scatterplots of the four datasets to see whether this conclusion holds.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(10,10))
for dataset, ax in zip(['I','II','III','IV'], axs.flatten()):
    x = anscombe_df[dataset, 'x']
    y = anscombe_df[dataset, 'y']
    sns.scatterplot(x=x, y=y, ax=ax)

> 1.11 Which (if any) assumption made by pearson correlation is violated in each of the four datasets?

---
Answer:

---

### _Optional_: Pearson correlation VS linear regression
The Anscombe's quartet demonstrates the pitfalls of blindly trusting Pearson correlation and summary statistics without taking a look at the actual data. Even when fitting a linear regression model to each dataset (drawing a best-fit straight line through the data) we obtain the same result in all four cases:

In [ ]:
sns.lmplot(x="x", y="y", col="dataset", hue="dataset", 
           data=sns.load_dataset("anscombe") , col_wrap=2)

Remembering again the definition of Pearson correlation:
> The Pearson correlation coefficient measures the **linear relationship**
    between two datasets. The calculation of the p-value relies on the assumption that each dataset is **normally distributed**.
 
### Spearman Correlation

So what do we do when the linearity assumptions of Pearson correlation fail? Let's look into other correlation methods.

> The Spearman correlation is a nonparametric measure of the monotonicity
    of the relationship between two datasets. Unlike the Pearson correlation,
    the Spearman correlation does not assume that both datasets are normally
    distributed. Like other correlation coefficients, this one varies
    between -1 and +1 with 0 implying no correlation. Correlations of -1 or
    +1 imply an exact monotonic relationship. Positive correlations imply that
    as x increases, so does y. Negative correlations imply that as x
    increases, y decreases.

So, the Spearman correlation coefficient measures monotonicity - i.e. do the two datasets increase together or descrease together. It doesn't matter anymore if the relationship is linear or not.

In [ ]:
for dataset in ['I','II','III','IV']:
    r, p_value = spearmanr(anscombe_df[dataset, 'x'], anscombe_df[dataset, 'y'])
    print(f'Dataset {dataset}: Spearman r = {round(r, 2)}, p value = {round(p_value, 2)}')

What if your datasets are correlated but the relationship is neither linear nor monotonic? Neither Pearson nor Spearman can tell you this!

In [ ]:
x = np.arange(-50, 50)
y = x**2 + 2*x + 2
sns.scatterplot(x=x, y=y)

pearson_coef, pearson_p = pearsonr(x, y)
print(f'Pearson r = {round(pearson_coef, 2)}, p value = {round(pearson_p, 2)}')

spearman_coef, spearman_p = spearmanr(x, y)
print(f'Spearman r = {round(spearman_coef, 2)}, p value = {round(spearman_p, 2)}')

### _Optional:_ correlation does not equal causation

One thing to note, while correlation metrics are useful to understand your dataset, they cannot be used to infer causal relationships. 

> 1.12 Take a look at the [Spurious Correlations](http://www.tylervigen.com/spurious-correlations) website for proof of the above statement. Discuss what you would need to do instead to infer a causal relationship.

---
Answer:

---

# 2. Dimensionality Reduction _(~45 minutes)_

In this notebook you will be introduced to two powerful techniques for working with high-dimensional datasets: Principal Component Analysis (PCA) and t-distributed Stochastic Neighbor Embedding (tSNE). Both approaches can roughly be categorized under the broad term dimensionality reduction. There are (many) more techniques for dimensionality reduction, and more applications than shown here. In this notebook we will focus on using dimensionality reduction for exploratory data analysis, and creating meaningfull plots of a high-dimensional dataset.

To start things off, we revisit the Iris dataset, which has four attributes.

In [ ]:
iris_df = pd.read_excel('../../datasets/iris_data.xlsx')

In [ ]:
iris_df.head()

In [ ]:
iris_df.describe()

Since we can't imagine or plot four dimensions at once, we could try to look at two dimensions at a time:

In [ ]:
# prepare two plot axes
fig, [ax1, ax2] = plt.subplots(ncols=2, figsize=(10,5))

# plot sepal length vs sepal width
sns.scatterplot(x="sepal length", y="sepal width", hue="species name", data=iris_df, ax=ax1)

# plot petal length vs petal width
sns.scatterplot(x="petal length", y="petal width", hue="species name", data=iris_df, ax=ax2)

While each pair shows some clustering based on species it's hard to tell how it all fits together. This becomes even harder with more dimensions/attributes.

> 2.1 For a hypothetical dataset of _N_ dimensions, how many pairwise scatterplots would you need to make to observe all possible combinations?

---
Answer:

---

### Principal Component Analysis (PCA)

Principal Component Analysis (PCA) is a procedure that finds a low dimensional representation of a high dimensional dataset. This is done in a specific way, such that the lower dimensional representation has a few nice properties:
    
- All low dimensional variables are linearly uncorrelated to each other
- All low dimensional variables are [orthogonal](https://en.wikipedia.org/wiki/Orthogonality) to each other
- The variables of the low dimensional representation are ordered by the amount of variation they capture in the original dataset

The variables of the low dimensional dataset are therefore referred to as the 'Principal Components'. By selecting the first few principal components you reduce the dimensionality of a dataset while maintaining the maximum possible amount of variation that was present in the original dataset.

> 2.2 To help get some intuition with what PCA does, execute the code below. This performs PCA on an example 3D dataset to get the first 2 principal components, which are highlighted as orange and green arrows. The 3D plot can be rotated by dragging. Describe in your own words how the three properties of PCA listed above can be seen from this example.

---
Answer:

---

In [ ]:
from pca_example import make_pca_example_figure
make_pca_example_figure()

---

You will now continue with performing PCA on some of the example datasets we have used before: `Iris`, `Boston` and `MNIST`. All of these datasets consist of more than two measured variables, so intuitively dimensionality reduction makes sense.

> _Optional_: Take a look at scikitlearn's PCA class definition.

In [ ]:
# sklearn PCA class definition
PCA??

Next, perform PCA on the iris dataset, keeping only the first two principal components.

In [ ]:
# Initialize a PCA transformer that will return the first two PCs as numpy array
pca = PCA(n_components=2)

# Transform/project the original 4D data into 2D
pcs = pca.fit_transform(iris_df[iris_df.columns[:-2]])
pcs.shape

> 2.3 Why is the shape of the returned numpy array 150x2?

---
Answer:

---

Let's plot our two principal components as a scatterplot, and color by species.

In [ ]:
sns.scatterplot(x=pcs[:, 0], y=pcs[:, 1], hue=iris_df["species name"])

> 2.4 What is different between the above scatterplot and the scatterplots for individual variables you made before?

---
Answer:

---

Similar as in the PCA example figure, it is possible to know for each prinipal component the amount of variance it explains in the original dataset. Most times,  the _ratio_ of explained variance is easiest to interpret, as it is expressed in percentages. 

In [ ]:
pca.explained_variance_ratio_

> 2.5 According to the explained variance ratio you calculated with above code, what percentage of variance in the original dataset is explained by your two principal components combined?

---
Answer:

---

> 2.6 How informative is the second principal component in terms of explained variance?

---
Answer:

---

## Nonlinear Dimensionality Reduction

Some of the properties of PCA (linearly uncorrelated PCs, orthogonal PCs) can prove to be too restrictive for real world datasets. Just like there exists linear and non-linear correlation, there exists linear and non-linear dimensionality reduction. 

PCA is linear dimensionality reduction, and the fact that you can calculate things like explained variance per component and component loadings are results of PCA being linear. 

Non-linear dimensionality reduction sacrifices these properties in favor of more informative low dimensional spaces. What is meant by more informative will become clear with the following example: performing dimensionality reduction on the `MNIST` dataset.

In [ ]:
mnist_dataset = datasets.load_digits()

In [ ]:
print(mnist_dataset.DESCR)

---

Instead of the usual named attributes, each data point in the MNIST dataset is an image representing a handwritten digit. These images are represented as a matrix of numbers:

In [ ]:
print(mnist_dataset.images[0])
print(mnist_dataset.images[0].shape)

Each number specifies the darkness of the pixel in that position:

In [ ]:
plt.imshow(mnist_dataset.images[0], cmap="Greys")

In [ ]:
plt.imshow(mnist_dataset.images[50], cmap="Greys")

Since each data point needs to be represented as a row in the dataframe, we simply flatten all the rows of the 8x8 matrix into one 64-length series of numbers

In [ ]:
mnist_dataset.data[0]

In [ ]:
mnist_df = pd.DataFrame(mnist_dataset.data)
mnist_df["digit"] = mnist_dataset.target

In [ ]:
mnist_df.head()

Let's first perform PCA on the `MNIST` dataset, reducing it from 64D to 2D. Making a scatterplot and coloring by digit should give some information on how well the original structure of the dataset is represented in 2D.

In [ ]:
# Reduce dimension to 2D with PCA
pca = PCA(n_components=2)
embedded_points = pca.fit_transform(mnist_df[mnist_df.columns[:-1]])

# Set some plotting parameters
f, ax = plt.subplots(1, figsize=(8,8))

# Make a scatterplot of first two principal components
sns.scatterplot(x=embedded_points[:, 0], y=embedded_points[:, 1],
                hue=mnist_df["digit"], legend="full", ax=ax,
                palette=sns.color_palette("hls", 10))

While there is some clustering by the digit represented by each image, it's not very clear and there is a lot of overlap. Looking at the explained variance ratios shows that this 2D PCA does not really capture a lot of the variance in the dataset either.

In [ ]:
pca.explained_variance_ratio_

Now let's try the non-linear dimensionality reduction method 't-distributed Stochastic Neighbour Embedding (t-SNE)'. The full details of t-SNE are outside of the scope of this course, so a very short summary will have to suffice here. In brief, t-SNE uses different distance metrics in the high dimensional data space and the low dimensional space, and uses iterative optimization to make all pairwise distances in both spaces as similar as possible. This iterative optimization makes that t-SNE can take a while to compute. 

It should be noted that, unlike PCA, t-SNE has several user-specified parameters that influence what the dimensionality reduction looks like. Whereas the default values are generally sensible starting points, it often pays to experiment a little with different parameter values.

In [ ]:
TSNE??

> ### _WARNING!_
> #### The code below runs tSNE, which is computationaly heavy and can take a while to complete.
> #### Contact a TA If you think tSNE is taking too long to finish.

In [ ]:
# Perform tSNE on MNIST to reduce from 64D to 2D
embedded_points = TSNE(n_components=2, init='pca', n_jobs=1).fit_transform(mnist_df[mnist_df.columns[:-1]])

# Some plotting parameters
f, ax = plt.subplots(1, figsize=(8,8))

# Scatterplot of low dimensional space
sns.scatterplot(x=embedded_points[:, 0], y=embedded_points[:, 1], hue=mnist_df["digit"], 
                legend="full", ax=ax, palette=sns.color_palette("hls", 10))

> 2.7 How do the 2D plots of the MNIST dataset differ between the PCA and tSNE dimensionality reduction? Which one do you think better represents the data?

---
Answer:

---

> 2.8 Try T-SNE on the Iris dataset. Is nonlinear dimensionality reduction needed in these cases?

---
Answer:

---

# 3. Clustering _(~45 minutes)_

In the Dimensionality Reduction notebook you may have already noticed that some datasets have distinct clusters. **Clustering** techniques go one step further and actively try to group data points into a number of groups / clusters such that data points in a group are more similar to other data points in the same group and dissimilar to the data points in other groups. Notice here that we don't specify what these groups are! Clustering is completely unsupervised, meaning that it looks for underlying structure in a dataset without relating it to any kind of external variable.

Some examples of clustering in real life:
* In bioinformatics, genes are often clustered according to their expression levels to find proteins which are potentially related to the same function or belong in the same pathway
* Clustering is used in social networks to recognize communities of people with common interests / behaviors
* Wikipedia has a nice [list of more clustering applications](http://en.wikipedia.org/wiki/Cluster_analysis#Applications)

We will look at two common clustering algorithms in this notebook.

# $K$-Means Clustering

$k$-means clustering aims to group *n* data points into *k* clusters, such that each data point or observation belongs to the cluster with the closest mean. The algorithm is described below

![clustering](kmeans_clustering.png)


> 3.1 What can happen if the initial "means" start in different positions?

---
Answer:

---

We'll first perform $k$-means on the Iris dataset

iris_df = pd.read_excel('../../datasets/iris_data.xlsx')
iris_df.head()

In [ ]:
# Initialize a sklearn kmeans clustering object
kmeans = cluster.KMeans(n_clusters=3, random_state=RANDOM_STATE)

# Cluster Iris data by calling the 'fit' function 
# Output will be stored in the kmeans object
kmeans.fit(iris_df.to_numpy()[:, :-1])

print('K-means labels:')
print(kmeans.labels_)

print('K-means cluster centers:')
print(kmeans.cluster_centers_)

> 3.2 Have a look at the _labels__ attribute of the kmeans object (code below). What does this array represent?

---
Answer:

---

There are two important attributes of this kmeans object that will come in useful:
* `kmeans.labels_` assigns a cluster number to each data point
* `kmeans.cluster_centers_` contains the centroids of each cluster (the final "means" after convergence)

> 3.3 Have a look at the _cluster_centers__ attribute of the kmeans object. Why is this a 3x4 matrix? What do the rows represent? What do the columns represent?

---
Answer:

---

> 3.4 How many points are present in each cluster? _Hint: To do this programmatically you can run `np.where(kmeans.labels_ == 0)` or `== 1` or `== 2`_

---
Answer:

---

We can see how the clusters look by plotting the data points and assigning a color to each cluster number. The black dots are the cluster centers.

In [ ]:
# dim1 and dim2 are column numbers, change to select different columns
dim1 = 0
dim2 = 1

sns.scatterplot(x=iris_df.iloc[:, dim1], y=iris_df.iloc[:, dim2], hue=kmeans.labels_, palette='deep')
sns.scatterplot(x=kmeans.cluster_centers_[:, dim1], y=kmeans.cluster_centers_[:, dim2], 
                color='black', s=200, alpha=0.7);

> 3.5 Do you think this is a good clustering?

---
Answer:

---

> 3.6 Try a different set of columns (flower properties), how does the clustering look?

---
Answer:

---

One interesting idea is to see whether these clusters, found just by looking at the feature values, correspond in any way to the species of each Iris flower. But note that in general, classes and clusters are two different notions: in principle a class can consist of different clusters, or different classes may form one large cluster. 

Below, we plot the same data points but this time colored by their species.

In [ ]:
sns.scatterplot(data=iris_df, x="sepal length", y="sepal width", hue="species name")

> 3.7 Can you find any differences by eye between the species labels and the cluster labels above? (Remember to compare the two plots using the same set of features)

---
Answer:

---

One disadvantage of $k$-means is that you have to provide the number of clusters as a parameter. Often, this is something you don't really know beforehand.

> 3.8 Repeat the clustering above but set the `n_clusters` parameter to a different number than 3 and plot the clusters. How do these clusters compare to the first set? Is there still any connection to the species label?

---
Answer:

---

## _OPTIONAL_: $K$-means clustering on the MNIST handwritten digits dataset

In [ ]:
mnist_dataset = datasets.load_digits()
mnist_dataframe = pd.DataFrame(mnist_dataset.data)
mnist_dataframe["digit"] = mnist_dataset.target

Remember, each data point is represented by 8x8=64 pixels:

In [ ]:
plt.imshow(mnist_dataset.images[1500], cmap="Greys")

Since we expect 10 kinds of digits in MNIST (the numbers 0 through 10), we can start out by trying to find 10 clusters

In [ ]:
n_clusters = 10

In [ ]:
kmeans = cluster.KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE)
kmeans.fit(mnist_dataset.data)

Using the concepts in the Dimensionality Reduction section, we'll first reduce the dataset into 2 dimensions so that we can visualize all the datapoints and cluster labels.

In [ ]:
# Perform t-SNE on the MNIST dataset to reduce it to 2 dimensions for plotting
tsne = TSNE(n_components=2, init='pca', n_jobs=1)
points = tsne.fit_transform(mnist_dataset.data)

In [ ]:
fig, ax = plt.subplots(1, figsize=(8,8))
sns.scatterplot(x=points[:, 0], 
                y=points[:, 1], 
                hue=kmeans.labels_, palette=sns.color_palette("hls", n_clusters), ax=ax)

As before, let's compare the clustering to the digit labels, and see if there's any correspondence.

In [ ]:
fig, ax = plt.subplots(1, figsize=(8,8))
sns.scatterplot(x=points[:, 0], 
                y=points[:, 1], 
                hue=mnist_dataframe["digit"], palette=sns.color_palette("hls", n_clusters), ax=ax)

> 3.9 Do you notice any clusters that are made up of more than one kind of digit? Remember, the color assignment for the clusters is random, so you cannot directly compare blue to blue.

---
Answer:

---

It's difficult to match up the clusters to their digits since there's so many of them. Instead, let's take a look at the cluster centers and see what kind of "digits" they form. Since the cluster centers are also just 64-length vectors of numbers, we can view them as an 8x8 image matrix just like the actual MNIST data.

In [ ]:
fig, ax = plt.subplots(2, 5, figsize=(10, 5))
centers = kmeans.cluster_centers_.reshape(10, 8, 8)
for axi, center in zip(ax.flat, centers):
    axi.set(xticks=[], yticks=[])
    axi.imshow(center, cmap="Greys")

> 3.10 Which digits do the cluster centers approximately correspond to?

---
Answer:

---

Another way to compare clusters and digits is by looking at the digits appearing in each cluster.

In [ ]:
# This makes a matrix that shows how many data 
# points representing a digit are in each cluster
mat = confusion_matrix(mnist_dataset.target, kmeans.labels_)
plt.figure(figsize=(10,7))
sns.heatmap(mat.T, square=True, annot=True, fmt='d', cbar=False,
            xticklabels=mnist_dataset.target_names,
            yticklabels=mnist_dataset.target_names)
plt.xlabel('digit label')
plt.ylabel('cluster label');

The rows represent each cluster while the columns represent each digit.

> 3.11 Which digits are often clustered together? 

---
Answer:

---

MNIST is a very visual dataset. Let's look at some examples of digits labeled as "8":

In [ ]:
n_images = len(mnist_dataset.images)
eights = [mnist_dataset.images[i] for i in range(n_images) if mnist_dataset.target[i] == 8]

fig, ax = plt.subplots(5, 5, figsize=(10, 10))
selected = [mnist_dataset.images[i] for i in range(len(mnist_dataset.images)) if mnist_dataset.target[i] == 8]
for axi, image in zip(ax.flat, selected):
    axi.set(xticks=[], yticks=[])
    axi.imshow(image, cmap="Greys")

> 3.12 Does it make sense that "8"s are often clustered together with "1"s? You can also take a look at some "1"s to compare (change 8 to 1 in the code above)

---
Answer:

---

> 3.13 Try choosing different numbers of clusters. If you use fewer than 10, which digits are merged into one cluster? If you use more than 10, which digits are split up into multiple clusters?

---
Answer:

---

## *OPTIONAL*: How many clusters?

In the previous examples we played around with the number of clusters and saw different results, but we had some grouping in mind already (3 Iris species, 10 MNIST digits). What if we have no idea how many clusters our data has? 

One popular approach is called the **Elbow** method. It uses the concept of cluster *inertia*, defined as the sum of squared distances of data points to their closest cluster center. As the number of clusters increases, the sum of squared distances tends to zero. Imagine we set `n_clusters` to its maximum value n (where n is number of data points) - each sample will form its own cluster and be its own cluster center, making the inertia 0. If we instead set `n_clusters` to 1, then there's just one big cluster with the center being the average of all the data points - hardly a clustering at all! So we need a compromise, which is what the Elbow method lets us find.

Let's plot the inertia value for $k$-means clustering with different numbers of clusters for the Iris dataset.

In [ ]:
inertias = []
ks = []
for k in range(1, 15):
    ks.append(k)
    kmeans = cluster.KMeans(n_clusters=k, random_state=RANDOM_STATE)
    kmeans.fit(iris_df[iris_df.columns[:-2]])
    inertias.append(kmeans.inertia_)

plt.figure(figsize=(10,5))
plt.plot(ks, inertias, 'bx-')
plt.xlabel('n_clusters')
plt.ylabel('Inertia')
plt.title('The Elbow Method showing the optimal n_clusters for the Iris dataset');

The optimal `n_clusters` occurs at the *elbow* in the plot (if you imagine the plot is an arm), i.e the point where the inertia stops decreasing drastically and starts leveling out.

> 3.14 What is the optimal `n_clusters` value for the Iris dataset according to this plot?

---
Answer:

---

Repeat the Elbow method for the MNIST dataset (by changing the `kmeans.fit` line above to fit on `mnist_dataset.data` as you did earlier)

> 3.15 What is the optimal `n_clusters` value for the MNIST dataset according to this method? Is this as clear-cut as it was for the Iris dataset?

---
Answer:

---

# Hierarchical clustering

Hierarchical clustering is another popular clustering method, quite different from $k$-means. It starts by treating each data point as a separate cluster. Then, it repeatedly performs the following two steps: 
    1. identify the two clusters that are closest together
    2. merge these two clusters
This continues until all the clusters are merged together. This is illustrated in the diagram below.
![Hierarchical Clustering](https://dashee87.github.io/images/hierarch.gif)

The actual output of hierarchical clustering is the dendogram on the right, which shows the similarity relationships between each data points to the others. Depending on where you decide to *cut* this dendogram horizontally (shown with a dotted line in the figure), you end up with a different number of clusters.

In the first point above, the algorithm has to decide which two clusters are closest together. This can be done in different ways, called **linkages**:

![Linkages](http://compbio.pbworks.com/f/linkages.JPG)

* Simple/Single Linkage - this defines the distance between two clusters as the distance between the two closest points (one from each cluster)
* Average Linkage - distance between clusters = distance between cluster centers
* Complete Linkage / Furthest Linkage - distance between clusters = distance between the two farthest points

This choice of linkage can drastically change the way clusters are formed.

In [ ]:
X_moons, y_moons = datasets.make_moons(n_samples=20, noise=0., random_state=RANDOM_STATE)
X_moons = pd.DataFrame(X_moons, columns=["feature_1", "feature_2"])

fig, ax = plt.subplots(figsize=(6,4))
sns.scatterplot(data=X_moons, x="feature_1", y="feature_2", hue=y_moons, ax=ax, s=100);

Think about how hierarchical clustering would work on the dataset above using 1) single 2) average and 3) complete linkage.

> 3.16 Which linkage type do you think would best suit this dataset?

---
Answer:

---

We'll now perform hierarchical clustering on the Iris dataset.

We'll start out with using average linkage for clustering. On the top you see the dendogram obtained by clustering the data points

In [ ]:
# Don't cluster on species name/identifier
iris_cluster_df = iris_df.iloc[:,:-2].T

sns.clustermap(iris_cluster_df, method="average", metric="euclidean",
             figsize=(12, 6), row_cluster=False)

Let's look at this with a distance matrix to understand why the dendogram is how it is.

In [ ]:
row_linkage = linkage(iris_cluster_df.T, method="average")
distance_matrix = squareform(pdist(iris_cluster_df.T))
sns.clustermap(distance_matrix, col_linkage=row_linkage, row_linkage=row_linkage,
               figsize=(10, 10))

> 3.17. Repeat the above clustering with single and complete linkage (change the `method` parameter above). What differences do you see?

---
Answer:

---

### _OPTIONAL_

Another option is to cluster the features rather than the data points. This allows us to see if there are any groups of features that behave similarly across data points.

In [ ]:
sns.clustermap(iris_cluster_df.T, method="average", metric="euclidean",
             figsize=(6, 12), row_cluster=False)

In [ ]:
row_linkage = linkage(iris_cluster_df, method="average")
distance_matrix = pd.DataFrame(squareform(pdist(iris_cluster_df)), 
                               index=iris_cluster_df.T.columns, columns=iris_cluster_df.T.columns)
sns.clustermap(distance_matrix, col_linkage=row_linkage, row_linkage=row_linkage,
               figsize=(4, 4))

> 3.18 Which features cluster together? Could you think of a reason for such a clustering?

---
Answer:

---

> 3.19 Repeat with single and complete linkage. What do you see?

---
Answer:

---